# Installation and quickstart

PyPFT computes the polar Fourier transform (PFT): the two-dimensional Fourier transform of a function sampled in polar coordinates, evaluated on a polar grid in the frequency domain. The transform is a chain of three discrete steps: an angular discrete Fourier transform (DFT), one discrete Hankel transform (DHT) per angular harmonic, and an inverse angular DFT.

$$f(r, \theta) \xrightarrow{\ \text{DFT}_\theta\ } f_n(r) \xrightarrow{\ \mathcal{H}_n\ } F_n(\rho) \xrightarrow{\ \text{IDFT}_\psi\ } F(\rho, \psi)$$

Here $f(r, \theta)$ is the function in space, at radius $r$ and angle $\theta$. $f_n(r)$ is its $n$-th angular harmonic, $\mathcal{H}_n$ is the Hankel transform of order $n$, and $F(\rho, \psi)$ is the transform in the frequency domain, at radius $\rho$ and angle $\psi$. The inverse PFT runs the same chain backwards.

Radially sampled magnetic resonance imaging (MRI) is one application. The scanner measures the image's Fourier transform along spokes through the center of k-space, which is a polar grid. The inverse PFT turns frequency-domain samples on a polar grid into an image on a polar grid.

## Install

PyPFT supports Python 3.12, 3.13, and 3.14.

### Optional: a virtual environment

A virtual environment keeps PyPFT and its dependencies separate from other projects. Create it once in your project folder, then activate it in each new terminal before you use it.

On macOS or Linux:

```bash
python3 -m venv .venv
source .venv/bin/activate
```

In Windows PowerShell:

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
```

In the Windows Command Prompt:

```bat
python -m venv .venv
.venv\Scripts\activate.bat
```

On Windows with the Python install manager (the `py` command), choose the interpreter version when you create the environment. Run `py install 3.12` first if that version is not installed yet, then activate the environment as shown above:

```powershell
py -3.12 -m venv .venv
```

With conda, create and activate a conda environment instead, then install PyPFT with `pip` as shown below:

```bash
conda create -n pypft python=3.12
conda activate pypft
```

### Install with pip

```bash
pip install pypft
```

If the `pip` command is not found, or you want to be sure the package goes into the active interpreter, run `pip` through Python itself:

```bash
python -m pip install pypft
```

To install from a clone of the repository, use an editable install. Changes you make to the clone then take effect without reinstalling:

```bash
git clone https://github.com/GhDamghani/PyPFT.git
cd PyPFT
pip install -e .
```

If you use [uv](https://docs.astral.sh/uv/), `uv add pypft` adds PyPFT to a uv project, and `uv pip install pypft` installs it into the active environment.

### Check the installation

In [ ]:
# Draw every figure below the cell that makes it
%matplotlib inline

# Read the installed version from the package metadata
from importlib.metadata import version

version(distribution_name="pypft")

## Quickstart

This tour runs the whole pipeline once. It builds a sampling grid, samples an image on it, transforms the samples forward and back, and plots the result. Each step gets its own notebook later, linked at the end of this one.

In [ ]:
# NumPy for arrays, matplotlib for figures, and PyPFT itself
import matplotlib.pyplot as plt
import numpy as np

import pypft

### An image

Any grayscale image works. Here it is two Gaussian blobs of different widths, placed away from the center so that the image is not circularly symmetric.

In [ ]:
# A 256 x 256 pixel image, with coordinates measured from its center
size = 256
rows, cols = np.mgrid[0:size, 0:size]
x = cols - size / 2  # grows to the right
y = rows - size / 2  # grows downward, like the row index

# A narrow blob right of the center plus a wider, dimmer blob above it
image = np.exp(-((x - 40) ** 2 + y**2) / (2 * 12**2))
image += 0.6 * np.exp(-(x**2 + (y + 50) ** 2) / (2 * 20**2))

# Show the image
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(X=image, cmap="gray")
ax.set_title(label="Cartesian image")
plt.show()

### The sampling grid

`pypft.PolarGrid` is the transform's own polar sampling grid. It has `n_radial` samples along the radius and `n_angular` samples around the circle. Its space limit `R` is the radius beyond which the function is taken to be zero. `R` is in the same unit as the image coordinates, which is pixels here. `pypft.check_adequacy` warns when `n_radial` is too small for `n_angular`. It stays silent for this grid.

In [ ]:
# 1024 radial x 31 angular samples inside a disk of radius 120 pixels
grid = pypft.PolarGrid(n_radial=1024, n_angular=31, R=120.0)

# Warns if n_radial is too small for n_angular; silent here
pypft.check_adequacy(grid=grid)

### Sampling the image

`pypft.sample_cartesian` evaluates the image at the grid's sample points. It returns one row per angle, with shape `(n_angular, n_radial)`. `forward_pft` expects PyPFT's `(radial, angular)` layout instead, so the result is transposed. Polar data that already comes on a uniform polar grid, with the same equally spaced radii on every spoke, goes through `pypft.resample_uniform_polar` instead, which interpolates it along each spoke onto the grid's points; [02_sampling_grids](02_sampling_grids.ipynb) shows it.

In [ ]:
# Interpolate the image at every grid point: one row per angle
samples = pypft.sample_cartesian(image=image, grid=grid)

# Transpose to the (radial, angular) layout the transform expects
f = samples.T
f.shape

### Forward and inverse transform

`pypft.forward_pft` takes $f(r, \theta)$ to $F(\rho, \psi)$ on the same grid, and `pypft.inverse_pft` takes it back. The round trip returns $f$ up to floating-point rounding. That shows the two directions are consistent. It does not show that $F$ is an accurate transform, which `05_pft_and_ipft` checks against a known analytic answer.

In [ ]:
# Space domain -> frequency domain
F = pypft.forward_pft(f=f, grid=grid)

# Frequency domain -> space domain
f_back = pypft.inverse_pft(F=F, grid=grid)

# Largest round-trip error, relative to the largest sample
float(np.max(a=np.abs(f_back - f)) / np.max(a=np.abs(f)))

### Plotting

A typed signal records which domain an array belongs to, and knows how to plot itself. `pypft.PolarSpatialSignal` holds $f$ and `pypft.PolarFrequencySignal` holds $F$. `pypft.plot_signal` draws a signal's magnitude and phase against its polar sample indices. `pypft.render_cartesian` draws a signal back onto a square pixel grid for display.

In [ ]:
# Wrap each array with its grid and its domain
spatial = pypft.PolarSpatialSignal(values=f, grid=grid)
frequency = pypft.PolarFrequencySignal(values=F, grid=grid)

# The spatial samples drawn as a Cartesian image, like the original
fig, ax = plt.subplots(figsize=(4, 4))
pypft.render_cartesian(signal=spatial, height=size, width=size, ax=ax)
plt.show()

# Magnitude and phase of f over the polar sample indices
pypft.plot_signal(signal=spatial)
plt.show()

# The same for F, zoomed in on the low frequencies that hold the smooth blobs' energy
for ax in pypft.plot_signal(signal=frequency):
    ax.set_ylim(bottom=-0.5, top=40)
plt.show()

## Where to go next

Each notebook assumes only the ones before it:

1. [01_polar_images_and_grids](01_polar_images_and_grids.ipynb) introduces sampling grids and moves an image between a Cartesian grid and a uniform polar grid.
2. [02_sampling_grids](02_sampling_grids.ipynb) introduces `PolarGrid`, the transform's own non-uniform grid.
3. [03_discrete_hankel_transform](03_discrete_hankel_transform.ipynb) introduces the discrete Hankel transform on its own.
4. [04_domains](04_domains.ipynb) introduces the four domains a signal passes through, and the typed signals above.
5. [05_pft_and_ipft](05_pft_and_ipft.ipynb) checks the forward and inverse PFT against a known analytic transform.
6. [06_batches](06_batches.ipynb) transforms many images in one call.
7. [07_pft_properties](07_pft_properties.ipynb) tours the transform's analytical properties.
8. [08_visualization](08_visualization.ipynb) covers tracing and plotting the pipeline.